# So sanh hai cach chia du lieu: RANDOM theo anh vs THEO SUBJECT

**Muc tieu:** huan luyen cung mot mo hinh CNN phan loai trang thai mat (eyes_closed / eyes_open) theo HAI cach chia train/val/test = 70/20/10, roi SO SANH ket qua.

- **Goc nhin 1 - RANDOM theo anh:** tron tat ca anh roi chia → cung mot nguoi co the xuat hien o ca train va test → RO RI DU LIEU (data leakage) → metric ao cao.
- **Goc nhin 2 - THEO SUBJECT:** chia theo nguoi (mot nguoi chi nam o mot tap) → trung thuc → phan anh kha nang tong quat that.

**Cong bang:** ca hai dung CHUNG cau hinh, kien truc, seed, class_weight; CHI khac ham chia.

**Truoc khi chay:** Runtime → Change runtime type → **GPU (T4)**. Upload `dataset.zip` (chua train/val/test/<lop>) len My Drive.

## Cell 1 — Cau hinh chung + GPU + co dinh seed

In [ ]:
import tensorflow as tf, numpy as np, random, os

# Co dinh seed o moi thu vien de ket qua tai lap duoc va so sanh CONG BANG
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

# Toan bo sieu tham so dung CHUNG cho ca hai thi nghiem (chi khac ham chia)
CONFIG = {
    'image_size': 64,        # kich thuoc anh dau vao
    'batch_size': 64,
    'epochs': 25,            # toi da; EarlyStopping se dung som
    'learning_rate': 1e-3,
    'dropout': 0.3,
    'split': {'train': 0.70, 'valid': 0.20, 'test': 0.10},
    'seed': SEED,
    'closed_class_weight': 1.3,   # he so uu tien recall lop eyes_closed (an toan)
    'aug': {'flip': 'horizontal', 'rotation': 0.05, 'zoom': 0.10, 'contrast': 0.20, 'brightness': 0.20},
}
CLASSES = ['eyes_closed', 'eyes_open']   # index 0 = eyes_closed, 1 = eyes_open
CONFIG

## Cell 2 — Mount Google Drive + giai nen dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, time
# >>> SUA dung ten file zip ban da upload len My Drive <<<
ZIP = '/content/drive/MyDrive/dataset.zip'
t = time.time()
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content')
SRC = '/content/dataset'   # ben trong co train/ val/ (hoac valid/) test/
print('Giai nen xong:', round(time.time() - t), 's ->', os.listdir(SRC))

## Cell 3 — Gom TOAN BO anh kem ma subject
Ten file MRL dang `s0021_00019_..._01.png`, trong do `s0021` la ma nguoi (subject). Ta trich ma nay de phuc vu cach chia theo subject.

In [ ]:
import re

all_items = []   # moi phan tu: (ten_lop, ma_subject, duong_dan_anh)
for split in ['train', 'val', 'valid', 'test']:      # gom tu moi thu muc con co san
    for c in CLASSES:
        d = f'{SRC}/{split}/{c}'
        if not os.path.isdir(d):
            continue
        for fn in os.listdir(d):
            m = re.match(r'(s\d+)', fn)              # bat tien to s####
            subj = m.group(1) if m else fn           # neu khong khop: coi moi file la 1 subject
            all_items.append((c, subj, os.path.join(d, fn)))

n_subj = len(set(s for _, s, _ in all_items))
print(f'Tong anh: {len(all_items)} | Tong subject: {n_subj}')

## Cell 4 — Hai HAM CHIA du lieu (diem khac biet duy nhat giua 2 goc nhin)

In [ ]:
from collections import defaultdict

def split_random(items, ratios, seed=SEED):
    """GOC NHIN 1: tron tat ca ANH roi cat 70/20/10.
    Hau qua: anh cua cung mot nguoi co the roi vao ca train lan test (RO RI)."""
    items = list(items)
    random.Random(seed).shuffle(items)               # tron theo tung anh
    n = len(items)
    n_tr = int(n * ratios['train'])
    n_va = int(n * ratios['valid'])
    return {'train': items[:n_tr], 'valid': items[n_tr:n_tr + n_va], 'test': items[n_tr + n_va:]}

def split_subject(items, ratios, seed=SEED):
    """GOC NHIN 2: chia theo NGUOI. Mot nguoi chi nam o dung mot tap → khong ro ri."""
    by = defaultdict(list)
    for it in items:
        by[it[1]].append(it)                         # gom anh theo subject (it[1])
    subs = list(by)
    random.Random(seed).shuffle(subs)                # tron danh sach NGUOI
    n = len(subs)
    n_tr = int(n * ratios['train'])
    n_va = int(n * ratios['valid'])
    groups = {'train': subs[:n_tr], 'valid': subs[n_tr:n_tr + n_va], 'test': subs[n_tr + n_va:]}
    out = {k: [] for k in groups}
    for k, ss in groups.items():
        for s in ss:
            out[k] += by[s]                          # dua TAT CA anh cua nguoi do vao dung 1 tap
    return out

## Cell 5 — Tao tf.data tu danh sach file + augmentation
Doc anh truc tiep tu duong dan (khong copy 84k anh) cho nhanh. Chuan hoa /255 mot lan.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
IMG = CONFIG['image_size']
a = CONFIG['aug']

# Khoi augmentation (chi ap dung cho tap train)
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(a['flip']),
    tf.keras.layers.RandomRotation(a['rotation']),
    tf.keras.layers.RandomZoom(a['zoom']),
    tf.keras.layers.RandomContrast(a['contrast']),
    tf.keras.layers.RandomBrightness(a['brightness'], value_range=(0.0, 1.0)),
], name='augmentation')

def make_ds(items, training):
    """Tao tf.data.Dataset tu danh sach (lop, subject, path).
    training=True: co shuffle + augmentation. training=False: KHONG shuffle (de y_true khop y_pred)."""
    paths  = [p for (c, s, p) in items]
    labels = tf.keras.utils.to_categorical([CLASSES.index(c) for (c, s, p) in items], num_classes=2)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 10000), seed=SEED, reshuffle_each_iteration=True)
    def load(path, label):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.resize(img, (IMG, IMG))
        img = tf.cast(img, tf.float32) / 255.0       # chuan hoa ve [0,1] - MOT lan duy nhat
        img.set_shape((IMG, IMG, 3))                 # gan shape tinh (decode_image lam mat shape)
        return img, label
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).batch(CONFIG['batch_size'])
    if training:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

## Cell 6 — Ham dung model + ham CHAY MOT THI NGHIEM (train + danh gia)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def build_model():
    """Kien truc CNN nhe: Conv32-BN-Pool -> Conv64-BN-Pool -> Conv128-BN-GAP -> Dropout -> Dense softmax."""
    return tf.keras.Sequential([
        tf.keras.layers.Input((IMG, IMG, 3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(CONFIG['dropout']),
        tf.keras.layers.Dense(2, activation='softmax'),
    ])

def run_experiment(split_fn, tag):
    """Chay tron mot thi nghiem voi ham chia split_fn. Tra ve dict ket qua + confusion matrix."""
    print('\n' + '=' * 64 + f'\n THI NGHIEM: {tag}\n' + '=' * 64)
    tf.keras.backend.clear_session()                 # xoa do thi cu, tranh trung ten layer
    tf.random.set_seed(SEED)                          # cong bang: cung khoi tao trong so

    # 1) Chia du lieu theo split_fn
    sp = split_fn(all_items, CONFIG['split'], SEED)
    counts = {k: {c: sum(1 for (cc, s, p) in v if cc == c) for c in CLASSES} for k, v in sp.items()}
    for k in ['train', 'valid', 'test']:
        print(f'  {k}: {counts[k]} (tong {sum(counts[k].values())})')

    # 2) Tao dataset (test KHONG shuffle de y_true khop y_pred)
    train_ds = make_ds(sp['train'], True)
    val_ds   = make_ds(sp['valid'], False)
    test_ds  = make_ds(sp['test'],  False)

    # 3) Tinh trong so lop (uu tien eyes_closed) tu tan suat tap train
    ntr = counts['train']; total = sum(ntr.values())
    cw = {0: (total / (2 * ntr['eyes_closed'])) * CONFIG['closed_class_weight'],
          1: (total / (2 * ntr['eyes_open']))}
    print('  class_weight:', {k: round(v, 3) for k, v in cw.items()})

    # 4) Dung va bien dich model
    model = build_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG['learning_rate']),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])
    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
    ]

    # 5) Huan luyen
    hist = model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['epochs'],
                     class_weight=cw, callbacks=cbs, verbose=1)

    # 6) Danh gia tren tap test
    probs  = model.predict(test_ds, verbose=0)
    y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds])   # nhan that
    y_pred = probs.argmax(1)                                              # nhan du doan
    rep = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, digits=4)
    cm  = confusion_matrix(y_true, y_pred)
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

    return {
        'tag': tag, 'epochs': len(hist.history['loss']),
        'train_acc': round(hist.history['accuracy'][-1], 4),
        'val_acc':   round(hist.history['val_accuracy'][-1], 4),
        'test_acc':  round(rep['accuracy'], 4),
        'recall_closed':    round(rep['eyes_closed']['recall'], 4),
        'precision_closed': round(rep['eyes_closed']['precision'], 4),
        'recall_open':      round(rep['eyes_open']['recall'], 4),
        'precision_open':   round(rep['eyes_open']['precision'], 4),
        'macro_f1': round(rep['macro avg']['f1-score'], 4),
        'cm': cm, 'history': hist.history,
    }

## Cell 7 — Chay GOC NHIN 1: RANDOM theo anh

In [ ]:
res_random = run_experiment(split_random, 'RANDOM theo anh (70/20/10)')

## Cell 8 — Chay GOC NHIN 2: THEO SUBJECT

In [ ]:
res_subject = run_experiment(split_subject, 'THEO SUBJECT (70/20/10)')

## Cell 9 — BANG SO SANH + 2 confusion matrix + dien giai

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import pandas as pd

# Bang so sanh
rows = []
for r in [res_random, res_subject]:
    rows.append({
        'Cach chia': r['tag'], 'Epochs': r['epochs'],
        'Train acc': r['train_acc'], 'Val acc': r['val_acc'], 'Test acc': r['test_acc'],
        'Macro F1': r['macro_f1'], 'Recall closed': r['recall_closed'],
        'Precision closed': r['precision_closed'], 'Recall open': r['recall_open'],
        'Gap train-val': round(r['train_acc'] - r['val_acc'], 4),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Muc do ro ri = chenh lech test acc giua hai cach chia
gap = res_random['test_acc'] - res_subject['test_acc']
print(f'\n>> Chenh lech test accuracy (random - subject) = {gap:+.4f}')
print('   -> Day la muc metric bi THOI PHONG do ro ri subject o cach chia random.')

# Hai confusion matrix canh nhau
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for i, r in enumerate([res_random, res_subject]):
    ConfusionMatrixDisplay(r['cm'], display_labels=CLASSES).plot(ax=ax[i], colorbar=False)
    ax[i].set_title(r['tag'], fontsize=10)
plt.tight_layout(); plt.savefig('so_sanh_confusion.png', dpi=200, bbox_inches='tight'); plt.show()

# Luu ket qua + tai ve
df.to_csv('so_sanh_split.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('so_sanh_split.csv')
files.download('so_sanh_confusion.png')

## Cell 10 — Bieu do so sanh (cot metric + duong cong validation)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ===== 1) Bieu do COT so sanh cac metric chinh giua 2 cach chia =====
metrics = ['test_acc', 'macro_f1', 'recall_closed', 'recall_open']
labels  = ['Test acc', 'Macro F1', 'Recall closed', 'Recall open']
rand_vals = [res_random[m] for m in metrics]
subj_vals = [res_subject[m] for m in metrics]

x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, rand_vals, w, label='RANDOM theo anh', color='#E8954B')
b2 = ax.bar(x + w/2, subj_vals, w, label='THEO SUBJECT',   color='#4C8BBF')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.80, 1.02); ax.set_ylabel('Gia tri')
ax.set_title('So sanh metric: RANDOM vs THEO SUBJECT')
ax.legend()
for b in list(b1) + list(b2):                       # ghi gia tri tren moi cot
    ax.annotate(f'{b.get_height():.3f}', (b.get_x() + b.get_width()/2, b.get_height()),
                ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig('so_sanh_metrics_bar.png', dpi=200, bbox_inches='tight'); plt.show()

# ===== 2) DUONG CONG validation cua 2 cach chia (accuracy & loss theo epoch) =====
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for r, col in [(res_random, '#E8954B'), (res_subject, '#4C8BBF')]:
    h = r['history']
    ax[0].plot(h['val_accuracy'], color=col, marker='o', ms=3, label=r['tag'])
    ax[1].plot(h['val_loss'],     color=col, marker='o', ms=3, label=r['tag'])
ax[0].set_title('Val accuracy theo epoch'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy'); ax[0].legend()
ax[1].set_title('Val loss theo epoch');     ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss');     ax[1].legend()
plt.tight_layout(); plt.savefig('so_sanh_curves.png', dpi=200, bbox_inches='tight'); plt.show()

# Tai 2 bieu do ve may de chen bao cao
from google.colab import files
files.download('so_sanh_metrics_bar.png')
files.download('so_sanh_curves.png')

## Cach doc ket qua
- **RANDOM theo anh**: test acc thuong CAO hon (≈98-99%) nhung la ket qua AO vi cung nguoi xuat hien o train+test (ro ri).
- **THEO SUBJECT**: test acc thap hon (≈96%) nhung TRUNG THUC vi danh gia tren nguoi moi.
- Dong `>> Chenh lech...` chinh la muc ro ri dinh luong → cau chot cho bao cao: *Random split cho so cao hon KHONG phai vi model tot hon ma do duoc test tren du lieu da thay.*
- Luu y: bo MRL chi co 37 nguoi nen tap test theo subject chi vai nguoi → ket qua co phuong sai; nen ghi vao muc Han che.